# Project 2: E-Commerce Business Dashboard
## Dataset: Brazilian E-Commerce Public Dataset by Olist
### 5 Business Questions This Project Answers
1. What are the overall KPIs: total revenue, total orders, average order value, return rate?
2. How does revenue grow month over month and what is the percentage change?
3. Which product categories drive the most revenue and which are declining?
4. Which customer geography contributes most to sales?
5. How do different customer segments compare on order value and frequency?

In [1]:
import pandas as pd

In [2]:
orders    = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items     = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments  = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
products  = pd.read_csv('../data/raw/olist_products_dataset.csv')

In [3]:
# Inspect each table
for name, df in [('orders', orders),
                 ('items', items),
                 ('payments', payments),
                 ('customers', customers),
                 ('products', products)]:
    print(f'\n{"="*40}')
    print(f'TABLE: {name}')
    print(f'Shape: {df.shape}')
    print(f'\nColumns: {df.columns.tolist()}')
    print(f'\nNull values:\n{df.isnull().sum()}')


TABLE: orders
Shape: (99441, 8)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Null values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

TABLE: items
Shape: (112650, 7)

Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Null values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

TABLE: payments
Shape: (103886, 5)

Columns: ['order_id', 'payment_sequential', 'pa

# Task 2: Data Cleaning & Merging Multiple Tables
What This Task Does
- Fix date columns across all tables
- Merge 5 tables into one master DataFrame
- Handle nulls and avoid row duplication from joins
- Engineer KPI columns: delivery days, order value, order month

### Fix Date Columns

In [4]:
print(orders.dtypes)

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


In [5]:
# Fix date columns in orders table
# Dates are stored as DD-MM-YYYY HH:MM (day-first); pandas' auto-inference picks the wrong
# format from the first (ambiguous) row and then errors on a later unambiguous one, so the
# format is given explicitly here.
date_format = '%d-%m-%Y %H:%M'
orders['order_purchase_timestamp']       = pd.to_datetime(orders['order_purchase_timestamp'], format=date_format)
orders['order_approved_at']              = pd.to_datetime(orders['order_approved_at'], format=date_format)
orders['order_delivered_carrier_date']   = pd.to_datetime(orders['order_delivered_carrier_date'], format=date_format)
orders['order_delivered_customer_date']  = pd.to_datetime(orders['order_delivered_customer_date'], format=date_format)
orders['order_estimated_delivery_date']  = pd.to_datetime(orders['order_estimated_delivery_date'], format=date_format)

# Also fix date in items table
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], format=date_format)

# Verify conversion worked
print(orders.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


### Aggregate Payments and Items

In [6]:
# Step 1 — Aggregate payments to one row per order
payments_agg = payments.groupby('order_id').agg(
    payment_value = ('payment_value', 'sum'),
    payment_type  = ('payment_type', 'first')
).reset_index()

print(f'payments_agg shape: {payments_agg.shape}')

# Step 2 — Aggregate items to one row per order
items_agg = items.groupby('order_id').agg(
    num_items     = ('order_item_id', 'count'),
    total_price   = ('price', 'sum'),
    total_freight = ('freight_value', 'sum')
).reset_index()

print(f'items_agg shape: {items_agg.shape}')

payments_agg shape: (99440, 3)
items_agg shape: (98666, 4)


### Merge All Tables

In [7]:
# Start with orders as the base table
df = orders.merge(customers[['customer_id', 'customer_state']],
                  on='customer_id',
                  how='left')

print(f'After customers merge: {df.shape}')

# Add payments
df = df.merge(payments_agg,
              on='order_id',
              how='left')

print(f'After payments merge: {df.shape}')

# Add items
df = df.merge(items_agg,
              on='order_id',
              how='left')

print(f'After items merge: {df.shape}')

After customers merge: (99441, 9)
After payments merge: (99441, 11)


After items merge: (99441, 14)


### Handle Nulls in Merged df

In [8]:
print('Nulls before handling:')
print(df.isnull().sum())

# payment_value/payment_type/num_items/total_price/total_freight are NaN only where the
# left-merge found no matching row in payments_agg or items_agg (order has no recorded
# payment or items) - fill with 0 / 'unknown' since the order legitimately has none
df['payment_value']  = df['payment_value'].fillna(0)
df['payment_type']   = df['payment_type'].fillna('unknown')
df['num_items']      = df['num_items'].fillna(0)
df['total_price']    = df['total_price'].fillna(0)
df['total_freight']  = df['total_freight'].fillna(0)

# order_approved_at / order_delivered_carrier_date / order_delivered_customer_date nulls are
# NOT merge artifacts - they reflect orders that never reached that stage (e.g. cancelled,
# unavailable). Leave as NaT so downstream delivery-time calculations stay accurate.

print(f'\ndf shape: {df.shape}')
print('\nNulls after handling:')
print(df.isnull().sum())
df.head()

Nulls before handling:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_state                      0
payment_value                       1
payment_type                        1
num_items                         775
total_price                       775
total_freight                     775
dtype: int64

df shape: (99441, 14)

Nulls after handling:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_state                      0
payment_value                       0
paymen

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,payment_value,payment_type,num_items,total_price,total_freight
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,SP,38.71,credit_card,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,BA,141.46,boleto,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,GO,179.12,credit_card,1.0,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-12-02 00:28:00,2017-12-15,RN,72.20,credit_card,1.0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,SP,28.62,credit_card,1.0,19.90,8.72


### Engineer KPI Columns

In [9]:
# delivery_days: days between purchase and customer delivery (NaT/NaN where never delivered)
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# order_value: actual amount paid for the order (revenue KPI)
df['order_value'] = df['payment_value']

# order_month: purchase month as a YYYY-MM string, for month-over-month growth analysis
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# is_delivered: whether the order actually reached the customer
df['is_delivered'] = df['order_status'] == 'delivered'

print(f'df shape: {df.shape}')
df.head()

df shape: (99441, 18)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,payment_value,payment_type,num_items,total_price,total_freight,delivery_days,order_value,order_month,is_delivered
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,SP,38.71,credit_card,1.0,29.99,8.72,8.0,38.71,2017-10,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,BA,141.46,boleto,1.0,118.70,22.76,13.0,141.46,2018-07,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,GO,179.12,credit_card,1.0,159.90,19.22,9.0,179.12,2018-08,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-12-02 00:28:00,2017-12-15,RN,72.20,credit_card,1.0,45.00,27.20,13.0,72.20,2017-11,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,SP,28.62,credit_card,1.0,19.90,8.72,2.0,28.62,2018-02,True


### Items with Category (separate item-level DataFrame for category analysis)
The aggregation step earlier collapsed `product_id` out of `df`, so category-level revenue
(business question 3) needs its own item-grain table built directly from `items` + `products`.

In [10]:
items_with_category = items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

print(f'items_with_category shape: {items_with_category.shape}')
items_with_category.head()

items_with_category shape: (112650, 8)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:00,58.90,13.29,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:00,239.90,19.93,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:00,199.00,17.87,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:00,12.99,12.79,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:00,199.90,18.14,ferramentas_jardim


### Save Clean Master DataFrame

In [11]:
df.to_csv('../data/processed/master.csv', index=False)

print(f'Saved master.csv — shape: {df.shape}')
df.head()

Saved master.csv — shape: (99441, 18)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,payment_value,payment_type,num_items,total_price,total_freight,delivery_days,order_value,order_month,is_delivered
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,SP,38.71,credit_card,1.0,29.99,8.72,8.0,38.71,2017-10,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,BA,141.46,boleto,1.0,118.70,22.76,13.0,141.46,2018-07,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,GO,179.12,credit_card,1.0,159.90,19.22,9.0,179.12,2018-08,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-12-02 00:28:00,2017-12-15,RN,72.20,credit_card,1.0,45.00,27.20,13.0,72.20,2017-11,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,SP,28.62,credit_card,1.0,19.90,8.72,2.0,28.62,2018-02,True


### Verification

In [12]:
print(f'master df shape: {df.shape}')
df.head()

master df shape: (99441, 18)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,payment_value,payment_type,num_items,total_price,total_freight,delivery_days,order_value,order_month,is_delivered
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,SP,38.71,credit_card,1.0,29.99,8.72,8.0,38.71,2017-10,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,BA,141.46,boleto,1.0,118.70,22.76,13.0,141.46,2018-07,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,GO,179.12,credit_card,1.0,159.90,19.22,9.0,179.12,2018-08,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-12-02 00:28:00,2017-12-15,RN,72.20,credit_card,1.0,45.00,27.20,13.0,72.20,2017-11,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,SP,28.62,credit_card,1.0,19.90,8.72,2.0,28.62,2018-02,True
